In [6]:
import numpy as np 
import pandas as pd    

df_sales = pd.read_csv(r"C:\Users\masri\Downloads\CSV_FILES\df_sales.csv")
df_products = pd.read_csv(r"C:\Users\masri\Downloads\CSV_FILES\df_products.csv")
print(df_sales)
print("\n\n---------------------------------------------------\n\n")
print(df_products)

     Product ID Salesperson ID Customer Region  Sales Date  Units Sold  \
0          P005           S003            West  2025-06-06          21   
1          P002           S002           North  2025-06-21         142   
2          P005           S001            West  2025-06-21         186   
3          P005           S005           North  2025-07-11         106   
4          P001           S005            East  2025-02-20         162   
...         ...            ...             ...         ...         ...   
4730       P005           S003            East  2025-03-30         198   
4731       P001           S002           North  2025-07-05         122   
4732       P003           S002            West  2025-04-29         189   
4733       P004           S001            West  2025-02-23         138   
4734       P005           S003           North  2025-02-28          43   

      Price per Unit  
0                 40  
1                 40  
2                 50  
3                 5

In [7]:
#Add a column to df_sales called Total Sales that contains the total sales of each row transaction.
df_sales["Total Sales"] = df_sales["Units Sold"] * df_sales["Price per Unit"] 
print(df_sales)
print("\n----------------------------------\n")
print(df_products)

     Product ID Salesperson ID Customer Region  Sales Date  Units Sold  \
0          P005           S003            West  2025-06-06          21   
1          P002           S002           North  2025-06-21         142   
2          P005           S001            West  2025-06-21         186   
3          P005           S005           North  2025-07-11         106   
4          P001           S005            East  2025-02-20         162   
...         ...            ...             ...         ...         ...   
4730       P005           S003            East  2025-03-30         198   
4731       P001           S002           North  2025-07-05         122   
4732       P003           S002            West  2025-04-29         189   
4733       P004           S001            West  2025-02-23         138   
4734       P005           S003           North  2025-02-28          43   

      Price per Unit  Total Sales  
0                 40          840  
1                 40         5680  
2  

In [13]:
#Merge these DataFrames together so that df_final contains the same data as df_products,
#but the second column contains all the corresponding product descriptions for the Product Id column using the column name from df_products.

# df_final = df_sales.merge(df_products, on= "Product ID", how = "left")

df_final = df_sales.merge(
    df_products,
    on="Product ID",
    how="left"
)
cols = ["Product ID", "Product Name"]+ [c for c in df_sales.columns if c != "Product ID"]
df_final = df_final[cols]
df_final

,Product ID,Product Name,Salesperson ID,Customer Region,Sales Date,Units Sold,Price per Unit,Total Sales
0,P005,Widget A,S003,West,2025-06-06,21,40,840
1,P002,Widget C,S002,North,2025-06-21,142,40,5680
2,P005,Widget A,S001,West,2025-06-21,186,50,9300
3,P005,Widget A,S005,North,2025-07-11,106,50,5300
4,P001,Widget B,S005,East,2025-02-20,162,30,4860
...,...,...,...,...,...,...,...,...
4730,P005,Widget A,S003,East,2025-03-30,198,30,5940
4731,P001,Widget B,S002,North,2025-07-05,122,40,4880
4732,P003,Gadget A,S002,West,2025-04-29,189,30,5670
4733,P004,Gadget B,S001,West,2025-02-23,138,40,5520


In [16]:
#Collect the data from df_sales to make a new DataFrame df_final_collected,
#that keeps as row indices the Product ID and Customer Region data and uses the Total Sales as a column.
#That is, we want to know for each of the five products and each of the four regions what the total Total Sales were.


df_final_collected = (
    df_sales
    .groupby(["Product ID", "Customer Region"])["Total Sales"]
    .sum()
    .to_frame()
)
print(df_final_collected)

                            Total Sales
Product ID Customer Region             
P001       East                  820720
           North                 847610
           South                 893400
           West                  855660
P002       East                  800120
           North                 822630
           South                 846170
           West                  761470
P003       East                  786700
           North                 882250
           South                 780490
           West                  787730
P004       East                  868170
           North                 743490
           South                 853150
           West                  873760
P005       East                  904190
           North                 693400
           South                 841520
           West                  849260


In [17]:
# Take the data from df_final_collected to make a new DataFrame df_final_regrouped that keeps as row index Product ID from df_final_collected.
#and moves the Customer Region data to a subindex of the column index. You have access to df_final_collected below.

df_final_regrouped = df_final_collected.unstack("Customer Region")
print(df_final_regrouped)

                Total Sales                        
Customer Region        East   North   South    West
Product ID                                         
P001                 820720  847610  893400  855660
P002                 800120  822630  846170  761470
P003                 786700  882250  780490  787730
P004                 868170  743490  853150  873760
P005                 904190  693400  841520  849260


In [20]:
#. Take the data from df_sales to make a new DataFrame df_cross consisting of the Salesperson ID data down the index 
#and the Product ID index across the column and each cross cell entry is the mean of the Price per Unit for each Salesperson ID by each Product ID. Note the name of the index and column below.

df_cross = df_sales.pivot_table(
    index = "Salesperson ID",
    columns="Product ID", 
    values= "Price per Unit",
    aggfunc= "mean"
)
print(df_cross)

Product ID           P001       P002       P003       P004       P005
Salesperson ID                                                       
S001            36.699029  34.152047  36.275510  35.444444  35.000000
S002            36.162162  35.384615  34.427861  35.614035  35.418994
S003            35.135135  34.345550  34.439024  35.478723  34.677419
S004            36.208791  34.019608  35.215054  34.734043  36.745562
S005            33.834951  34.500000  34.870466  36.250000  34.882629


In [22]:
df_melted = df_sales.drop(columns=["Total Sales"]).melt(
    id_vars = ["Product ID", "Salesperson ID", "Customer Region", "Sales Date"],
    value_vars = ["Units Sold", "Price per Unit"],            
    var_name= "Unital Information",
    value_name = "Unital Value"
)
print(df_melted)

     Product ID Salesperson ID Customer Region  Sales Date Unital Information  \
0          P005           S003            West  2025-06-06         Units Sold   
1          P002           S002           North  2025-06-21         Units Sold   
2          P005           S001            West  2025-06-21         Units Sold   
3          P005           S005           North  2025-07-11         Units Sold   
4          P001           S005            East  2025-02-20         Units Sold   
...         ...            ...             ...         ...                ...   
9465       P005           S003            East  2025-03-30     Price per Unit   
9466       P001           S002           North  2025-07-05     Price per Unit   
9467       P003           S002            West  2025-04-29     Price per Unit   
9468       P004           S001            West  2025-02-23     Price per Unit   
9469       P005           S003           North  2025-02-28     Price per Unit   

      Unital Value  
0     